# Table State: финальный инференс (раздел 15.1 контракта)

Воспроизводит оба отмеченных на Kaggle сабмита командой `run.py` из кэшей источников. Интернет выключен, GPU не нужен.

| файл | конфиг | Public |
|---|---|---|
| `VLM_hands_avg` (основной, пишется в `submission.csv`) | `configs/submissions/VLM_hands_avg.yaml` | 0.75957 |
| `VLM_items` (вторая отметка) | `configs/submissions/VLM_items.yaml` | 0.74357 |

**Add Input**

| вход | что берётся |
|---|---|
| данные соревнования (`rascar-hack-case-2`) или их копия `throddy/rascar-hack-case-2` | пакет организаторов: таблицы, кадры, `baseline.py` |
| `throddy/table-state-code` | код репозитория (`run.py`, `ts/`, конфиги, сохранённые сабмиты для сверки) |
| `victimestsate/ts-artifacts-dets` | `dets/ft_yolo__v1` (свой детектор, руки) |
| `throddy/ts-artifacts-ov-pseudo` | `rowfeats/ov_counts__v1` (счётчики Grounding DINO и OWLv2) |
| вывод ноутбуков `throddy/ts-det-vlm-shards-0-1` и `throddy/notebook15b164940c` | ответы Qwen3-VL-4B по шардам 0–3 (`_parts/vlm_qwen__v1/shard_*_of_4.jsonl`); здесь они склеиваются в `rowfeats/vlm_qwen__v1` без запуска модели |

Пересчёт источников на GPU: `tracks/det/README.md`, `tracks/ov/README.md`, `tracks/det/vlm/run_vlm.py`. Слаги в коде не зашиты: входы находятся по содержимому.


In [ ]:
import hashlib, os, re, shutil, subprocess, sys
from pathlib import Path

CONFIGS = ['configs/submissions/VLM_hands_avg.yaml',   # первый пишется в /kaggle/working/submission.csv
           'configs/submissions/VLM_items.yaml']
EXPECTED_GIT_SHA = ''   # коммит кода, которым получены сабмиты; если указан, датасет с другим кодом отклоняется
INPUT = Path(os.environ.get('TS_KAGGLE_INPUT', '/kaggle/input'))
WORKING = Path(os.environ.get('TS_KAGGLE_WORKING', '/kaggle/working'))


def find(name):
    """Все файлы с таким именем во входах. Ссылки внутри входов проходятся, но каждый каталог только один раз
    (защита от циклов), а найденный файл должен лежать внутри /kaggle/input или в смонтированном датасете Kaggle."""
    seen, found = set(), []
    for root, dirs, files in os.walk(INPUT, followlinks=True):
        key = (os.stat(root).st_dev, os.stat(root).st_ino)
        if key in seen:
            dirs[:] = []
            continue
        seen.add(key)
        for f in files:
            if (name(f) if callable(name) else f == name):
                path = Path(root) / f
                real = path.resolve()
                assert real.is_relative_to(INPUT.resolve()) or str(real).startswith(('/kaggle/', '/mnt/')), f'Файл вне входов: {real}'
                found.append(path)
    return sorted(found)


print('входы:', sorted(str(p.relative_to(INPUT)) for p in INPUT.glob('*/*'))[:30])

# 1. Данные соревнования: каталог с train_labels.csv, tables.csv и data_card.json (как в baseline.ipynb организаторов).
data = [p.parent for p in find('train_labels.csv')
        if (p.parent / 'tables.csv').is_file() and (p.parent / 'data_card.json').is_file()]
assert len(data) == 1, f'Нужен ровно один каталог данных соревнования, найдено: {data}'

# 2. Код: каталог с run.py, ts/ и всеми конфигами. Копируется в рабочую папку (входы только для чтения).
code = sorted({p.parent for p in find('run.py')
               if (p.parent / 'ts' / 'stack.py').is_file() and all((p.parent / c).is_file() for c in CONFIGS)})
assert len(code) == 1, f'Нужен ровно один датасет с кодом, где есть {CONFIGS}; найдено: {code}'
code_sha = (code[0] / 'GIT_SHA').read_text().strip() if (code[0] / 'GIT_SHA').is_file() else 'нет файла GIT_SHA'
print('код:', code[0], '| коммит:', code_sha)
assert not EXPECTED_GIT_SHA or code_sha == EXPECTED_GIT_SHA, f'В датасете код коммита {code_sha}, ожидался {EXPECTED_GIT_SHA}'
repo = WORKING / 'table-state'
shutil.rmtree(repo, ignore_errors=True)
shutil.copytree(code[0], repo, symlinks=True, ignore=shutil.ignore_patterns('__pycache__', '.git', 'artifacts', 'data'))

# 3. Артефакты всех конфигов собираются из подключённых входов в одну папку.
import yaml
artifacts = WORKING / 'artifacts'
shutil.rmtree(artifacts, ignore_errors=True)
env = {**os.environ, 'TS_DATA': str(data[0]), 'TS_ARTIFACTS': str(artifacts)}
needed = set()
for config in CONFIGS:
    cfg = yaml.safe_load((repo / config).read_text(encoding='utf-8'))
    needed |= {('dets', s) for s in cfg['sources'].get('dets', []) if s != 'prod'}
    needed |= {('rowfeats', s) for s in cfg['sources'].get('rowfeats', [])}
for kind, name in sorted(needed):
    assert re.fullmatch(r'[A-Za-z0-9_]+__v\d+', name), f'Недопустимое имя источника в конфиге: {name!r}'
    (artifacts / kind).mkdir(parents=True, exist_ok=True)
    shards = [p for p in find(lambda f: re.fullmatch(r'shard_\d+_of_\d+\.jsonl', f) is not None)
              if p.parent.name == name and p.parent.parent.name == '_parts']
    if kind == 'rowfeats' and name.startswith('vlm_qwen__v') and shards:
        # Ответы VLM лежат в выводе ноутбуков шардов: склеиваем их так же, как run_vlm.py --merge, без запуска модели.
        total = {int(re.search(r'_of_(\d+)', p.name).group(1)) for p in shards}
        index = sorted(int(re.search(r'shard_(\d+)_', p.name).group(1)) for p in shards)
        assert len(total) == 1 and index == list(range(total.pop())), f'{name}: нужен полный набор шардов без повторов, найдено: {shards}'
        parts = artifacts / 'rowfeats' / '_parts' / name
        parts.mkdir(parents=True, exist_ok=True)
        for p in shards:
            shutil.copyfile(p, parts / p.name)
            print(f'{kind}/_parts/{name}/{p.name}  <-  {p}')
        version = name.split('__v')[1]
        subprocess.run([sys.executable, 'tracks/det/vlm/run_vlm.py', '--subset', 'all', '--of', str(len(index)),
                        '--version', version, '--merge'], cwd=repo, env=env, check=True)
        continue
    for ext in ('.parquet', '.json'):
        found = [p for p in find(name + ext) if p.parent.name == kind]
        assert len(found) == 1, f'Нужен ровно один файл {kind}/{name}{ext} в подключённых входах, найдено: {found}'
        shutil.copyfile(found[0], artifacts / kind / (name + ext))
        print(f'{kind}/{name}{ext}  <-  {found[0]}')

# 4. Одна команда воспроизводимого запуска на каждый конфиг.
outputs = {}
for i, config in enumerate(CONFIGS):
    out = WORKING / ('submission.csv' if i == 0 else f'submission_{Path(config).stem}.csv')
    subprocess.run([sys.executable, 'run.py', '--data', str(data[0]), '--artifacts', str(artifacts),
                    '--config', config, '--out', str(out), '--from-cache'], cwd=repo, env=env, check=True)
    outputs[config] = out
    print(Path(config).stem, '->', out, 'sha256', hashlib.sha256(out.read_bytes()).hexdigest())


In [ ]:
# Сверка с сохранёнными сабмитами из датасета с кодом: счётчики должны совпасть точно, occ_prob до 1e-9.
import pandas as pd
for config, out in outputs.items():
    name = Path(config).stem
    saved = repo / 'submissions' / f'{name}.csv'
    assert saved.is_file(), f'{saved.name} нет в датасете с кодом: сверка обязательна'
    new, old = pd.read_csv(out), pd.read_csv(saved)
    assert len(new) == len(old) == 4733 and (new.row_id == old.row_id).all(), f'{name}: строки не совпали'
    for column in new.columns[1:]:
        diff = (new[column] - old[column]).abs()
        print(f'{name:<14} {column:<9} расхождений {(diff > 1e-9).sum():>4}, макс. {diff.max():.3g}')
        assert (diff > 1e-9).sum() == 0, f'{name}: {column} не совпал с сохранённым сабмитом'
    print(f'{name}: совпадает с submissions/{name}.csv (occ_prob до 1e-9: ExtraTrees бейзлайна складывает деревья в два потока)')
